In [9]:
import os
import glob
import base64
from PIL import Image
import nbformat


In [20]:
import os
import glob
import base64
from PIL import Image
import nbformat

def get_image_info(filepath):
    try:
        with Image.open(filepath) as img:
            width, height = img.size
        return width, height
    except Exception as e:
        return None, None

def encode_image_base64(filepath):
    with open(filepath, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode()

def collect_images_from_folder(folder):
    image_files = []
    for ext in ('*.png', '*.jpg', '*.jpeg', '*.gif'):
        # Use recursive=True to search subfolders
        image_files.extend(glob.glob(os.path.join(folder, '**', ext), recursive=True))
    return image_files

def extract_images_from_notebooks(notebook_folder):
    images = []
    for nb_path in glob.glob(os.path.join(notebook_folder, "*.ipynb")):
        nb = nbformat.read(nb_path, as_version=4)
        for cell in nb.cells:
            if cell.cell_type == 'code' and 'outputs' in cell:
                for output in cell['outputs']:
                    if output.output_type == 'display_data':
                        for mime, data in output.get('data', {}).items():
                            if mime.startswith('image/'):
                                images.append({
                                    'source': nb_path,
                                    'mime': mime,
                                    'data': data
                                })
    return images

In [21]:
# Collect images from images/ folder
folder_images = collect_images_from_folder('/content/drive/MyDrive/Berkley/mlai/capstone_project/images')

# Collect images embedded in notebooks (assuming notebooks are in current folder or specify path)
notebook_images = extract_images_from_notebooks('/content/drive/MyDrive/Berkley/mlai/capstone_project/notebooks')



In [22]:
# Start HTML report
html = "<html><head><title>Image Summary Report</title></head><body>"
html += "<h1>Images Summary Report</h1>"

html += "<h2>Images from images/ folder</h2>"
for img_path in folder_images:
    width, height = get_image_info(img_path)
    img_b64 = encode_image_base64(img_path)
    html += f"<div><b>{img_path}</b> ({width}x{height})<br>"
    html += f'<img src="data:image/png;base64,{img_b64}" height="100"/></div><br>'

html += "<h2>Images embedded in notebooks</h2>"
for idx, img in enumerate(notebook_images):
    html += f"<div><b>Notebook:</b> {img['source']}<br>"
    html += f'<img src="data:{img["mime"]};base64,{img["data"]}" height="100"/></div><br>'

html += "</body></html>"

os.makedirs("/content/drive/MyDrive/Berkley/mlai/capstone_project/reports", exist_ok=True)
with open("/content/drive/MyDrive/Berkley/mlai/capstone_project/reports/image_summary.html", "w") as f:
    f.write(html)

print("Report generated: /content/drive/MyDrive/Berkley/mlai/capstone_project/reports/image_summary.html")

Report generated: /content/drive/MyDrive/Berkley/mlai/capstone_project/reports/image_summary.html


In [23]:
# Calculate the number of images from each source
num_folder_images = len(folder_images)
num_notebook_images = len(notebook_images)

# Provide a summary
print("Image Summary Report:")
print(f"- Total images found in the specified folder and its subfolders: {num_folder_images}")
print(f"- Total images found embedded in the outputs of the specified notebooks: {num_notebook_images}")

# Optional: Provide more details about the notebook images if available
if num_notebook_images > 0:
    print("\nDetails about embedded notebook images:")
    # You could add code here to analyze MIME types or sources if needed
    # For a brief summary, we'll just list the count for now.
    pass

Image Summary Report:
- Total images found in the specified folder and its subfolders: 20
- Total images found embedded in the outputs of the specified notebooks: 20

Details about embedded notebook images:


analyze image content using the Gemini API.

In [30]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

try:
    GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
except userdata.SecretNotFoundError:
    print("API key not found. Please store your Google API key in Colab secrets with the name GOOGLE_API_KEY.")
except Exception as e:
    print(f"An error occurred configuring the API: {e}")

initialize the Gemini Vision model.

In [35]:
try:
    # Update the model name to a recommended alternative like gemini-1.5-flash
    gemini_model = genai.GenerativeModel('gemini-1.5-flash-latest')
    print("Gemini model initialized successfully.")
except Exception as e:
    print(f"An error occurred initializing the model: {e}")
    gemini_model = None # Set to None so we can check before using

Gemini model initialized successfully.


Now, iterate through the collected images and use the Gemini model to describe their content. start with the images collected from the folder and then proceed to the images embedded in the notebooks. Due to potential API rate limits or processing time, a limited number of images are processed

In [38]:
from IPython.display import display, Image as IPyImage
import tempfile # Import tempfile for creating temporary files

if 'gemini_model' in locals() and gemini_model is not None:
    print("Analyzing images from folder:")
    # Process all folder images
    if folder_images:
        for i, img_path in enumerate(folder_images):
            try:
                print(f"\nAnalyzing {img_path}:")
                # Display the image
                display(IPyImage(filename=img_path, width=100))
                img = Image.open(img_path)
                response = gemini_model.generate_content(["Describe the content of this image.", img])
                print(response.text)
            except Exception as e:
                print(f"Could not analyze image {img_path}: {e}")
    else:
        print("No folder images found to analyze.")


    print("\nAnalyzing images embedded in notebooks:")
    # Process all notebook images
    if notebook_images:
        for i, img_data in enumerate(notebook_images):
            try:
                print(f"\nAnalyzing image from notebook {img_data['source']}:")
                # Display the image
                # For notebook images, data is already base64 encoded
                display(IPyImage(data=base64.b64decode(img_data['data']), width=100))

                # Attempt to open the image data by saving to a temporary file first
                decoded_image_data = base64.b64decode(img_data['data'])
                with tempfile.NamedTemporaryFile(delete=False, suffix='.png') as temp_img_file: # Use .png as a common suffix
                    temp_img_file.write(decoded_image_data)
                    temp_img_path = temp_img_file.name

                img = Image.open(temp_img_path)

                response = gemini_model.generate_content(["Describe the content of this image.", img])
                print(response.text)

                # Clean up the temporary file
                os.remove(temp_img_path)

            except Exception as e:
                print(f"Could not analyze image from notebook {img_data['source']}: {e}")
    else:
        print("No notebook images found to analyze.")

else:
    print("Gemini model not initialized. Please check the previous steps.")

Output hidden; open in https://colab.research.google.com to view.